# E15-B — `e15_new`'s generation, from `final.pt` (nothing is scored here)

Registration: `research/e15/PREREG_E15.md` §§9.7, 13. Run this **after the laptop has
verified the training**. It writes 50 files at E12's seeds with the adopted decode
(raw + D1 + D1′), so each cell pairs with `e12_d1p`'s files, the frozen baseline.

**Attach two inputs:** **`uie-e15-code`**, and the **E15B_train** notebook's **output**.
**Settings:** GPU T4, Internet **On**. **Save Version → Save & Run All.** About 2 h.
A rerun keeps every finished file with its record. When done, download
`/kaggle/working/e15_gen`.

In [ ]:
# --- 1. locate the code bundle -----------------------------------------------------
import hashlib, json
from pathlib import Path


def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()


root = Path("/kaggle/input")
code_hits = sorted(root.rglob("E15_CODE_MANIFEST.json")) if root.exists() else []
if not code_hits:
    raise SystemExit("Attach the uie-e15-code dataset (right panel -> Add Input), then rerun.")
CODE = code_hits[0].parent
WORK = Path("/kaggle/working")
print("code:", CODE)

train_prov = sorted(p for p in root.rglob("PROVENANCE.json")
                    if (p.parent / "e15_new").is_dir()) if root.exists() else []
if not train_prov:
    raise SystemExit("Attach the E15B_train notebook's output too, then rerun.")
TRAIN = train_prov[0].parent
tprov = json.loads(train_prov[0].read_text())
CKPT = TRAIN / "e15_new" / "final.pt"
assert CKPT.is_file() and sha256(CKPT) == tprov["arms"]["e15_new"]["final_sha256"], \
    "e15_new: final.pt is not the one E15B_train recorded"
print("checkpoint verified:", CKPT)

In [ ]:
# --- 2. environment: install first, then run THIS code and prove it ------------
import importlib, subprocess, sys
from importlib import metadata

TLDEXTRACT = "5.3.1"          # the version the local evaluation parses with


def installed(dist):
    try:
        return metadata.version(dist)
    except metadata.PackageNotFoundError:
        return None


if installed("tldextract") != TLDEXTRACT:
    try:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                        f"tldextract=={TLDEXTRACT}"], check=True)
    except subprocess.CalledProcessError:
        raise SystemExit("pip could not install tldextract: Settings -> Internet -> On, rerun.")
    importlib.invalidate_caches()
assert installed("tldextract") == TLDEXTRACT, installed("tldextract")

# CODE ONLY on the path: the data bundle carries an older `uie`.
for p in (CODE / "research" / "e15", CODE / "research" / "e13", CODE / "research" / "native", CODE):
    sys.path.insert(0, str(p))
for name in [m for m in sys.modules if m == "uie" or m.startswith("uie.")
             or m in ("e8_common", "e11_common", "e12_common", "e13_common", "e15_common")]:
    del sys.modules[name]
import uie
import e12_common as K
import e15_common as E15
manifest = json.loads((CODE / "E15_CODE_MANIFEST.json").read_text())
for mod in (uie, K, E15):
    assert str(CODE) in mod.__file__, f"{mod.__name__} is SHADOWED -- check sys.path"
bad = [f for f, h in manifest["files"].items() if sha256(CODE / f) != h]
bad += [t for t, h in manifest["tables"].items() if sha256(CODE / t) != h]
if sha256(CODE / "e12_final.pt") != manifest["e12_final"]:
    bad.append("e12_final.pt")
if sha256(CODE / E15.STREAM_FILE) != manifest["stream"]["sha256"]:
    bad.append(E15.STREAM_FILE)
assert not bad, f"STOP. files differ from the manifest: {bad}"
print(f"code: commit {manifest['commit'][:12]}; {len(manifest['files'])} files, "
      f"{len(manifest['tables'])} tables, e12_final.pt and the stream ({manifest['stream']['rows']:,} rows) verified")

import torch
if not torch.cuda.is_available():
    raise SystemExit("No GPU. Settings -> Accelerator -> GPU T4, then rerun.")
GPU = torch.cuda.get_device_name(0)
print("torch:", torch.__version__, "| gpu:", GPU)

In [ ]:
# --- 3. tables, the registered jobs, the model ---------------------------------------
from uie.controlled import StateSuffixTables
from uie.suffixes import SuffixTable
from uie.train import load_checkpoint

shipped = SuffixTable.load(CODE / "suffix_table.json")
tables = StateSuffixTables.load(CODE / "state_suffix_tables.json")
presence_slash = json.loads((CODE / "presence_slash_table.json").read_text())["table"]
device = torch.device("cuda")
OUT = WORK / "e15_gen"
OUT.mkdir(parents=True, exist_ok=True)
ORDER = [("free",), ("control",)]
jobs = sorted(E15.jobs(), key=lambda j: (0 if j["mode"] == "free" else 1, j["label"], j["state"] or "", j["g"]))
print(f"{len(jobs)} files, {sum(j['n'] for j in jobs):,} URLs; decode {E15.decode_kwargs()}")
CKPT_SHA = {"e15_new": sha256(CKPT)}
PURPOSE = "E15-B (research/e15/PREREG_E15.md): e15_new, raw + D1 + D1'; generation only"
model, tok, _ = load_checkpoint(CKPT, device=device)
model.to(device).eval()
assert tok.presence_plan and tok.root_slash and tok.slash_plan and tok.vocab_size == 110
print(f"loaded e15_new: vocab {tok.vocab_size}, max_len {model.cfg.max_len}")

In [ ]:
# --- 4. generate -----------------------------------------------------------------------
# PROVENANCE.json is rewritten after EVERY file; a rerun keeps a file only with a
# matching record -- anything else is made again: same seed, same bytes.
import platform
import time

PROV = OUT / "PROVENANCE.json"
prov = json.loads(PROV.read_text()) if PROV.exists() else {"files": {}}
prov.update({"purpose": PURPOSE, "checkpoint_file": "final.pt",
             "code_commit": manifest["commit"], "checkpoints": CKPT_SHA,
             "tables": manifest["tables"],
             "environment": {"torch": torch.__version__, "python": platform.python_version(),
                             "gpu": GPU, "tldextract": metadata.version("tldextract")}})
t_start = time.time()
done = 0
for j in jobs:
    path = OUT / j["file"]
    rec = prov["files"].get(j["file"])
    if rec and path.is_file() and sha256(path) == rec["sha256"]:
        done += 1
        continue
    t0 = time.time()
    trace = []
    if j["mode"] == "free":
        urls = E15.free_urls(model, tok, j["label"], j["g"], shipped, presence_slash,
                             device=device, plan_trace=trace)
    else:
        urls = E15.control_urls(model, tok, j["label"], j["state"], j["g"], tables, presence_slash,
                                device=device, plan_trace=trace)
    assert len(urls) == j["n"], (j["file"], len(urls))
    path.write_text(json.dumps(urls))
    prov["files"][j["file"]] = {**{k: v for k, v in j.items() if k != "file"},
                                "sha256": sha256(path), "seconds": round(time.time() - t0, 1),
                                "plan_trace_mean": sum(trace) / len(trace) if trace else None}
    PROV.write_text(json.dumps(prov, indent=1))
    done += 1
    el = (time.time() - t_start) / 60
    print(f"  {done:>3}/{len(jobs)}  {j['file']:<46} {time.time() - t0:6.0f}s  "
          f"({el:.0f} min so far)", flush=True)
print(f"\nall {len(jobs)} files written. Now download {OUT}.")